## Imports

In [8]:

%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
import math
import torch
import einops
import pprint as pp
from torch import nn
from pathlib import Path
from torchdiffeq import odeint
from argparse import Namespace
import plotly.graph_objects as go

top_dir = Path.cwd().parent
sys.path.insert(0, str(top_dir.absolute()))

from src import *


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load ODE from model checkpoint or initialize randomly

In [9]:
device = "cuda:0"

checkpoint_path = top_dir / 'checkpoints' / 'sst_moe_lstm_cnn_best.pt'
loaded_vals = torch.load(checkpoint_path, weights_only=False)
sensors = loaded_vals['sensors']
args = Namespace(**loaded_vals['args'])

model = models.MixedModel(args)
model.load_state_dict(loaded_vals['model_state_dict'])
model.to(device)

sindy_coefficients = model.encoder.get_dense_sindy_coefficients()

print("SINDy coefficients shape:", [cf.shape for cf in sindy_coefficients])

None

SINDy coefficients shape: [torch.Size([5, 5]), torch.Size([5, 5]), torch.Size([5, 5]), torch.Size([5, 5]), torch.Size([5, 5]), torch.Size([5, 5]), torch.Size([5, 5]), torch.Size([5, 5]), torch.Size([5, 5]), torch.Size([5, 5])]


## Load dataset

In [10]:
train_ds, val_ds, test_ds, metadata = datasets.load_dataset(args)
print("Training dataset length: ", len(train_ds))

None


Training dataset length:  1091


## Load model state before SINDy layer

In [11]:

input_window = train_ds[0][0][0:args.input_length].to(device)

input_sensors = []
for sensor in sensors:
    input_sensors.append(input_window[:, sensor[0], sensor[1], :])
input_sensors = torch.stack(input_sensors, dim=2)
input_sensors = einops.rearrange(input_sensors, "w n d -> 1 w (n d)")

x = model.encoder.lstm(input_sensors)[1][0][-1] # 1 x hidden

print("pre_sindy shape:", x.shape)
print("min:   ", x.min().item())
print("max:   ", x.max().item())
print("median:", x.median().item())
print("mean:  ", x.mean().item())
print("std:   ", x.std().item())

None


pre_sindy shape: torch.Size([1, 5])
min:    -0.5584548711776733
max:    0.4657113254070282
median: -0.09932602941989899
mean:   -0.09224735200405121
std:    0.3738391399383545


## Custom coefficients

In [ ]:

if 0:
    experts = nn.ModuleList(
        [
            sindy_layer.SindyLayer(
                d_model=args.hidden_size,
                forecast_length=args.forecast_length,
                device=device,
                strict_symmetry=args.strict_symmetry,
                std_init=2.,
            )
            for _ in range(args.n_experts)
        ]
    )
    model.encoder.experts = experts

    sindy_coefficients = model.encoder.get_dense_sindy_coefficients()


## Print coefficients

In [13]:
print("Eigv real parts:")
pp.pprint([eig.real for eig in model.encoder.get_sindy_layer_coefficients_eigenvalues()])
print("Eigv imag parts:")
pp.pprint([eig.imag for eig in model.encoder.get_sindy_layer_coefficients_eigenvalues()])
model.encoder.print_sindy_layer_coefficients()


Eigv real parts:
[tensor([ 0.0000e+00, -1.8626e-08, -4.3212e-07, -1.6231e-07, -2.9052e-07]),
 tensor([ 0.0000e+00,  5.9954e-09,  1.8976e-08, -2.3190e-07,  1.3926e-07]),
 tensor([ 0.0000e+00,  2.6616e-07, -4.0965e-08, -4.6729e-07, -4.3638e-07]),
 tensor([ 0.0000e+00, -2.3797e-07, -9.7775e-09,  8.9420e-08, -1.1926e-07]),
 tensor([ 4.7684e-07, -5.3644e-07,  1.5421e-09,  2.2468e-07,  1.7155e-08]),
 tensor([-4.7684e-07, -1.1921e-06, -2.3738e-07, -5.7047e-08,  4.2380e-08]),
 tensor([ 0.0000e+00,  1.1921e-07,  5.4393e-08, -4.9259e-08, -4.6197e-07]),
 tensor([ 0.0000e+00, -2.3210e-09, -1.0431e-07,  2.7844e-07,  2.3847e-08]),
 tensor([ 0.0000e+00,  9.5637e-07,  4.4681e-07, -5.9528e-08,  2.4623e-07]),
 tensor([ 0.0000e+00,  0.0000e+00, -1.0862e-07, -5.0105e-07, -5.0665e-07])]
Eigv imag parts:
[tensor([-10.6988,   6.2418,  -3.2153,  -0.0818,   2.2372]),
 tensor([-8.8288, -4.3995, -0.1840,  3.9782,  3.0196]),
 tensor([-5.1276,  0.3134,  3.0993,  5.7302,  6.1523]),
 tensor([-7.0338,  1.1947,  0.582

## ODE forward


In [20]:
#forecast_length = args.forecast_length
forecast_length = 100
print("Forecast length: ", forecast_length)

rollouts = [helpers.ode_forward(x, sc, forecast_length)[0] for sc in sindy_coefficients]

print("Rollout shape:", [r.shape for r in rollouts])


Forecast length:  100
Rollout shape: [torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5])]


## Plot

In [22]:
for rollout in rollouts:
   plots.plot_ode_forward(rollout, title="ODE forward rollout", figsize=(1000, 1000))
